<a href="https://colab.research.google.com/github/Rhedonderon/toddler_nutrition_app/blob/main/ToddlerNutrition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Readme: The code takes twelve large files from the USDA page, filters to use only the columns needed, then merge and clean data until I have two datasets to filter further into a condensed list intended for a high fidelity mock up for a project about baby and toddler nutrition in an effort to address modern childhood nutrition conserns and build a portfolioof work.

### Environment Setup & Storage

In [ ]:
# Import the Google Colab drive module
from google.colab import drive

# Mount Google Drive to access the raw USDA data files
# Note: This will prompt for authentication and only needs to be run once per session.
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Library Imports

In [ ]:
# import pandas to manipulate data, cleaning,and analysis
import pandas as pd

### File Path Configuration

In [ ]:
# Base paths for the USDA FoodData Central datasets.
# By defining paths as variables here, file locations are separated from the processing logic.

# --- SR Legacy Files (April 2018) ---
food_legacy = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_sr_legacy_food_csv_2018-04/food.csv'
food_nutrient_legacy = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_sr_legacy_food_csv_2018-04/food_nutrient.csv'
# nutrient_legacy and food_category_legacy doesn't have fcd_id
nutrient_legacy = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_sr_legacy_food_csv_2018-04/nutrient.csv'
food_category_legacy = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_sr_legacy_food_csv_2018-04/food_category.csv'
food_portion_legacy = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_sr_legacy_food_csv_2018-04/food_portion.csv'

# --- Foundation Foods (December 2025) ---
food = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_foundation_food_csv_2025-12-18/food.csv'
food_nutrient = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_foundation_food_csv_2025-12-18/food_nutrient.csv'
# nutrient and food_category doesn't have fcd_id
nutrient = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_foundation_food_csv_2025-12-18/nutrient.csv'
food_category = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_foundation_food_csv_2025-12-18/food_category.csv'
food_portion = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_foundation_food_csv_2025-12-18/food_portion.csv'

# --- Branded foods (December 2025) ---
branded_food = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_branded_food_csv_2025-12-18/branded_food.csv'
food_nutrient_branded = '/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_branded_food_csv_2025-12-18/food_nutrient.csv'

### Define File Preview Function

In [ ]:
def preview_file(filename):
    """
    Reads and displays the first 5 rows of a CSV file.

    This function allows for fast schema exploration
    (column names, data shapes, and inferred data types) without
    loading massive datasets entirely into system memory.
    """

    # Read only the first 5 rows to preserve RAM
    preview = pd.read_csv(filename, nrows=5)

    # Print a clear separator so the output is readable
    print(f"\n--- PREVIEWING: {filename} ---")

    # Print the data
    print(preview)

    # Print the column names
    print("\nCOLUMNS FOUND:")
    print(preview.columns.tolist())

### Master File List Collection

In [ ]:
""" Create a master list containing all
the individual file paths, so they can
be easily looped through for batch
processing or previewing."""


all_files = [
food_legacy,
food_nutrient_legacy,
nutrient_legacy,
food_category_legacy,
food_portion_legacy,
food,
food_nutrient,
nutrient,
food_category,
food_portion,
branded_food,
food_nutrient_branded
]

### Automated Batch Preview

In [ ]:
# Loop through all the files in the master list
for f in all_files:
    try:
        # nrows limits the number of rows pulled to prevent a crash
        preview = pd.read_csv(f, nrows=5)
        print(f"Success: {f}")
        print(preview_file(f))
    except Exception as e:
        # If a file is missing or corrupted, catch the error and keep going
        print(f"Could not read {f}: {e}")

Success: /content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_sr_legacy_food_csv_2018-04/food.csv

--- PREVIEWING: /content/drive/MyDrive/Toddler_Nutrition/00_Incoming/FoodData_Central_sr_legacy_food_csv_2018-04/food.csv ---
   fdc_id       data_type                                        description  \
0  167512  sr_legacy_food  Pillsbury Golden Layer Buttermilk Biscuits, Ar...   
1  167513  sr_legacy_food  Pillsbury, Cinnamon Rolls with Icing, refriger...   
2  167514  sr_legacy_food  Kraft Foods, Shake N Bake Original Recipe, Coa...   
3  167515  sr_legacy_food     George Weston Bakeries, Thomas English Muffins   
4  167516  sr_legacy_food         Waffles, buttermilk, frozen, ready-to-heat   

   food_category_id publication_date  
0                18       2019-04-01  
1                18       2019-04-01  
2                18       2019-04-01  
3                18       2019-04-01  
4                18       2019-04-01  

COLUMNS FOUND:
['fdc_id', 'data_type', 'de

### Variable For Targeted Columns

In [ ]:

# Pre-defining target columns for filtering data.
# This prevents loading unnecessary information, keeping the data frames lean.

# --- SR Legacy Columns ---
food_legacy_cols = ['fdc_id', 'description', 'food_category_id']
food_nutrient_legacy_cols = ['fdc_id', 'nutrient_id', 'amount']
# nutrient_legacy_cols and food_category_legacy doesn't have fcd_id
nutrient_legacy_cols = ['id', 'name', 'unit_name']
food_category_legacy_cols = ['description', 'id']
food_portion_legacy_cols = ['fdc_id', 'amount', 'modifier']

# --- Foundation Foods Columns ---
food_cols = ['fdc_id', 'description', 'food_category_id']
food_nutrient_cols = ['fdc_id', 'nutrient_id', 'amount']
# nutrient and food_category doesn't have fdc_id
nutrient_cols = ['id', 'name', 'unit_name']
food_category_cols = ['description', 'id']
food_portion_cols = ['fdc_id', 'amount', 'modifier']

# --- Branded Foods Columns ---
branded_food_cols = ['fdc_id', 'gtin_upc', 'brand_owner', 'ingredients', 'short_description']
food_nutrient_branded_cols = ['fdc_id', 'nutrient_id', 'amount']

### Variable For Reading Columns

In [ ]:
"""
Load datasets into memory.
Passing pre-defined column lists into 'usecols' drastically reduced
memory overhead by only importing the necessary information.
"""

# --- Load Legacy Files ---
df_food_legacy = pd.read_csv(food_legacy, usecols=food_legacy_cols)
df_food_nutrient_legacy = pd.read_csv(food_nutrient_legacy, usecols=food_nutrient_legacy_cols)
df_nutrient_legacy = pd.read_csv(nutrient_legacy, usecols=nutrient_legacy_cols)
df_food_category_legacy = pd.read_csv(food_category_legacy, usecols=food_category_legacy_cols)
df_food_portion_legacy = pd.read_csv(food_portion_legacy, usecols=food_portion_legacy_cols)

# --- Load Foundation Files ---
df_food = pd.read_csv(food, usecols=food_cols)
df_food_nutrient = pd.read_csv(food_nutrient, usecols=food_nutrient_cols)
df_nutrient = pd.read_csv(nutrient, usecols=nutrient_cols)
df_food_category = pd.read_csv(food_category, usecols=food_category_cols)
df_food_portion = pd.read_csv(food_portion, usecols=food_portion_cols)

# --- Branded Food ---
df_branded_foods = pd.read_csv(branded_food, usecols=branded_food_cols, low_memory=False)
df_food_nutrient_branded = pd.read_csv(food_nutrient_branded, usecols=food_nutrient_branded_cols)

### Merge Foundational Data

In [ ]:
# Start by merging food and food_nutrient data from foundational files
merged_data = pd.merge(df_food, df_food_nutrient, on='fdc_id', how='left')

# Merge with df_nutrient to get nutrient names and units
# Assuming 'id' in df_nutrient corresponds to 'nutrient_id' in merged_data
merged_data = pd.merge(
    merged_data,
    df_nutrient,
    left_on='nutrient_id',
    right_on='id',
    how='left',
    suffixes=('', '_nutrient_info')
)

# Rename columns for clarity and drop the redundant 'id' from df_nutrient
merged_data.rename(columns={'name': 'nutrient_name', 'unit_name': 'nutrient_unit'}, inplace=True)
merged_data.drop(columns=['id'], axis=1, inplace=True)

# Merge with df_food_category to get food category descriptions
# Assuming 'id' in df_food_category corresponds to 'food_category_id' in merged_data
merged_data = pd.merge(
    merged_data,
    df_food_category,
    left_on='food_category_id',
    right_on='id',
    how='left',
    suffixes=('', '_category_info')
)

# Rename column for clarity and drop the redundant 'id' from df_food_category
merged_data.rename(columns={'description_category_info': 'food_category_description'}, inplace=True)
merged_data.drop(columns=['id'], axis=1, inplace=True)


# Merge with df_food_portion to add portion information
# This merge is on 'fdc_id'
merged_data = pd.merge(
    merged_data,
    df_food_portion.rename(columns={'amount': 'portion_amount', 'modifier': 'portion_modifier'}),
    on='fdc_id',
    how='left'
)

# verify results of the merge
print(merged_data.head())
print(merged_data.columns)

   fdc_id            description  food_category_id  nutrient_id  amount  \
0  319874  HUMMUS, SABRA CLASSIC              16.0          NaN     NaN   
1  319875  HUMMUS, SABRA CLASSIC              16.0          NaN     NaN   
2  319876  HUMMUS, SABRA CLASSIC              16.0          NaN     NaN   
3  319877                 Hummus              16.0       1051.0   56.30   
4  319877                 Hummus              16.0       1002.0    1.28   

  nutrient_name nutrient_unit    food_category_description  portion_amount  \
0           NaN           NaN  Legumes and Legume Products             NaN   
1           NaN           NaN  Legumes and Legume Products             2.0   
2           NaN           NaN  Legumes and Legume Products             NaN   
3         Water             G  Legumes and Legume Products             NaN   
4      Nitrogen             G  Legumes and Legume Products             NaN   

  portion_modifier  
0              NaN  
1              NaN  
2              Na

### Merging Legacy Files

In [ ]:
# Start by merging food and food_nutrient data from the legacy files
df_merged_legacy = pd.merge(df_food_legacy, df_food_nutrient_legacy, on='fdc_id', how='left')

# Merge with df_nutrient_legacy to get nutrient names and units
df_merged_legacy = pd.merge(
    df_merged_legacy,
    df_nutrient_legacy,
    left_on='nutrient_id',
    right_on='id',
    how='left',
    suffixes=('', '_nutrient_info_legacy')
)

# Rename columns for clarity and drop the redundant 'id' from df_nutrient_legacy
df_merged_legacy.rename(columns={'name': 'nutrient_name', 'unit_name': 'nutrient_unit'}, inplace=True)
df_merged_legacy.drop(columns=['id'], axis=1, inplace=True)

# Merge with df_food_category_legacy to get food category descriptions
df_merged_legacy = pd.merge(
    df_merged_legacy,
    df_food_category_legacy,
    left_on='food_category_id',
    right_on='id',
    how='left',
    suffixes=('', '_category_info_legacy')
)

# Rename column for clarity and drop the redundant 'id' from df_food_category_legacy
df_merged_legacy.rename(columns={'description_category_info_legacy': 'food_category_description'}, inplace=True)
df_merged_legacy.drop(columns=['id'], axis=1, inplace=True)


# Merge with df_food_portion_legacy to add portion information
df_merged_legacy = pd.merge(
    df_merged_legacy,
    df_food_portion_legacy.rename(columns={'amount': 'portion_amount', 'modifier': 'portion_modifier'}),
    on='fdc_id',
    how='left'
)

# verify results of the merge
print("Merged Legacy Data (first 5 rows):")
print(df_merged_legacy.head())
print("Merged Legacy Data Columns:")
print(df_merged_legacy.columns)

Merged Legacy Data (first 5 rows):
   fdc_id                                        description  \
0  167512  Pillsbury Golden Layer Buttermilk Biscuits, Ar...   
1  167512  Pillsbury Golden Layer Buttermilk Biscuits, Ar...   
2  167512  Pillsbury Golden Layer Buttermilk Biscuits, Ar...   
3  167512  Pillsbury Golden Layer Buttermilk Biscuits, Ar...   
4  167512  Pillsbury Golden Layer Buttermilk Biscuits, Ar...   

   food_category_id  nutrient_id   amount         nutrient_name nutrient_unit  \
0                18         1003     5.88               Protein             G   
1                18         1007     3.50                   Ash             G   
2                18         1062  1286.00                Energy            kJ   
3                18         1079     1.20  Fiber, total dietary             G   
4                18         1089     2.12              Iron, Fe            MG   

  food_category_description  portion_amount portion_modifier  
0            Baked Products   

### Combining Foundation and Legacy Data

In [ ]:
# Concatenate the foundation (merged_data) and the newly created (df_merged_legacy)
combined_data = pd.concat([merged_data, df_merged_legacy], ignore_index=True)

# Verify the results
print("Combined Data (first 5 rows):")
print(combined_data.head())
print("Total rows in combined data after dropping duplicates:", len(combined_data))
print(combined_data.columns)

Combined Data (first 5 rows):
   fdc_id            description  food_category_id  nutrient_id  amount  \
0  319874  HUMMUS, SABRA CLASSIC              16.0          NaN     NaN   
1  319875  HUMMUS, SABRA CLASSIC              16.0          NaN     NaN   
2  319876  HUMMUS, SABRA CLASSIC              16.0          NaN     NaN   
3  319877                 Hummus              16.0       1051.0   56.30   
4  319877                 Hummus              16.0       1002.0    1.28   

  nutrient_name nutrient_unit    food_category_description  portion_amount  \
0           NaN           NaN  Legumes and Legume Products             NaN   
1           NaN           NaN  Legumes and Legume Products             2.0   
2           NaN           NaN  Legumes and Legume Products             NaN   
3         Water             G  Legumes and Legume Products             NaN   
4      Nitrogen             G  Legumes and Legume Products             NaN   

  portion_modifier  
0              NaN  
1       

### Merge Branded Files

In [ ]:
# merge branded foods files
merged_branded = pd.merge(
    df_branded_foods,
    df_food_nutrient_branded,
    left_on='fdc_id',
    right_on='fdc_id',
    how='left'
)

# Rename columns for clarity
merged_branded.rename(columns={'brand_owner': 'brand_name', 'short_description': 'food_description', 'amount': 'nutrient_amount'}, inplace=True)

# Verify the results
print("merged_branded columns: ", merged_branded.columns.tolist())
print(merged_branded.head())

merged_branded columns:  ['fdc_id', 'brand_name', 'gtin_upc', 'ingredients', 'food_description', 'nutrient_id', 'nutrient_amount']
    fdc_id                                brand_name        gtin_upc  \
0  1105904  Richardson Oilseed Products (US) Limited  00027000612323   
1  1105904  Richardson Oilseed Products (US) Limited  00027000612323   
2  1105904  Richardson Oilseed Products (US) Limited  00027000612323   
3  1105904  Richardson Oilseed Products (US) Limited  00027000612323   
4  1105904  Richardson Oilseed Products (US) Limited  00027000612323   

     ingredients food_description  nutrient_id  nutrient_amount  
0  Vegetable Oil              NaN       1257.0             0.00  
1  Vegetable Oil              NaN       1293.0            53.33  
2  Vegetable Oil              NaN       1253.0             0.00  
3  Vegetable Oil              NaN       1092.0             0.00  
4  Vegetable Oil              NaN       1008.0           867.00  


### Define Core Foods List

In [ ]:
# save and define list of foods for sample prototype. Run once to save as 'master key'
# subject to change upon further research
core_foods = ["apples, raw",
            "bananas",
            "blueberries, raw",
            "grapes",
            "strawberries, raw",
            "pear",
            "Apple and pear",
            "carrots",
            "sweet potato",
            "green beans",
            "butternut squash",
            "Spinach",
            "broccoli",
            "chicken breast",
            "chicken nugget",
            "beans, black",
            "eggs",
            "peanut butter",
            "tofu",
            "oatmeal",
            "brown rice",
            "pasta",
            "cracker",
            "cereal",
            "whole wheat bread",
            "honey",
            "cookies",
            "potato chips",
            "fries",
            "mac and cheese",
            "ice cream",
            "milk, human",
            "formula",
            "apple juice",
            "hemp seeds",
            "yogurt melts",
            "Babyfood, snack, GERBER",
            "whole milk",
            "cheddar cheese",
            "cottage cheese",
            "yogurt"
            ]

### Search Foundation & Legacy Data For Matches

In [ ]:
# Initialize lists
# Found matching foods
results = []

# food without matched after searching first dataframe
to_find = []

# foods that wasn't found during any searches
missing_foods = []

for food in core_foods:
    # search foundation/Legacy foods first
    # head(1) captures first result. Keeps results lean and targeted.
    match = combined_data[combined_data['description'].str.contains(food, case=False, na=False)].head(1)

    # update lists
    if not match.empty:
        results.append(match)

    else:
        to_find.append(food)

### Verify Current Results

In [ ]:
# Show all the found foods to ensure they are correct
print(results)
print(len(results))

[        fdc_id                                        description  \
223267  167793  Apples, raw, fuji, with skin (Includes foods f...   

        food_category_id  nutrient_id  amount             nutrient_name  \
223267               9.0       1257.0     0.0  Fatty acids, total trans   

       nutrient_unit food_category_description  portion_amount  \
223267             G   Fruits and Fruit Juices             1.0   

       portion_modifier  
223267      cup, sliced  ,         fdc_id        description  food_category_id  nutrient_id  amount  \
115584  790647  BANANAS, OVERRIPE               9.0          NaN     NaN   

       nutrient_name nutrient_unit food_category_description  portion_amount  \
115584           NaN           NaN   Fruits and Fruit Juices             NaN   

       portion_modifier  
115584              NaN  ,          fdc_id       description  food_category_id  nutrient_id  amount  \
145032  2263889  Blueberries, raw               9.0       1063.0   9.356   

   

### Print Items Not Found

In [ ]:
# Shows remaining food items to be found
print(to_find)
print(len(to_find))

['mac and cheese', 'hemp seeds']
2


### Search Branded Files

In [ ]:
# search remaining food items in to_find list in merged_branded file

branded_filtered = merged_branded.copy()

for food in to_find:

    # Search specifically column brand name or food description
    match_branded_name = merged_branded[merged_branded['brand_name'].str.contains(food, case=False, na=False)].head(1)
    match_branded_description = merged_branded[merged_branded['food_description'].str.contains(food, case=False, na=False)].head(1)

    # update lists
    if not match_branded_name.empty:
        results.append(match_branded_name)

    elif not match_branded_description.empty:
        results.append(match_branded_description)

    else:
        missing_foods.append(food)

### Check For Missing Food

In [ ]:
print(missing_foods)
print(len(missing_foods))

[]
0


### Review Final Results

In [ ]:
print(results)
print(len(results))

[        fdc_id                                        description  \
223267  167793  Apples, raw, fuji, with skin (Includes foods f...   

        food_category_id  nutrient_id  amount             nutrient_name  \
223267               9.0       1257.0     0.0  Fatty acids, total trans   

       nutrient_unit food_category_description  portion_amount  \
223267             G   Fruits and Fruit Juices             1.0   

       portion_modifier  
223267      cup, sliced  ,         fdc_id        description  food_category_id  nutrient_id  amount  \
115584  790647  BANANAS, OVERRIPE               9.0          NaN     NaN   

       nutrient_name nutrient_unit food_category_description  portion_amount  \
115584           NaN           NaN   Fruits and Fruit Juices             NaN   

       portion_modifier  
115584              NaN  ,          fdc_id       description  food_category_id  nutrient_id  amount  \
145032  2263889  Blueberries, raw               9.0       1063.0   9.356   

   

### Consolodating Results

In [ ]:
# Makes loose rows into a single structured dataset
# 'ignore_index=True' drops the original chaotic row numbers from the source datasets
# and builds a clean, sequential index starting from 0.
toddler_foods = pd.concat(results, ignore_index=True)

### File Export

In [ ]:
# Send it to your spreadsheet
# By setting 'index=False', we prevent Pandas from writing a nameless first row-number column,
# ensuring the file perfectly aligns with standard formatting for downstream tools like Tableau.

toddler_foods.to_csv('/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/target_foods.csv', index=False)

### Verify File Contents

In [ ]:
# This reads the file
verify_file = pd.read_csv('/content/drive/MyDrive/Toddler_Nutrition/00_Incoming/target_foods.csv')

# This prints the file for verifying quality
print(verify_file)

     fdc_id                                        description  \
0    167793  Apples, raw, fuji, with skin (Includes foods f...   
1    790647                                  BANANAS, OVERRIPE   
2   2263889                                   Blueberries, raw   
3   2263890                         Grapes, red, seedless, raw   
4    327526  Strawberries, Raw, Pass 2, Region 2, n/a, Pota...   
5    324318  Pickles, kosher dill, spears, VLASIC (FL,MO) -...   
6    173026  Fruit cocktail, (peach and pineapple and pear ...   
7    326197  Carrots, sliced or crinkle cut, frozen, unprep...   
8   2346404    Sweet potatoes, orange flesh, without skin, raw   
9   2346644                                   Green beans, raw   
10   173510     Babyfood, vegetable, butternut squash and corn   
11  1750352                                      Spinach, baby   
12   321612                  Broccoli, raw (IN1,NY1) - CY0906E   
13   331899  Chicken breast, non-enhanced, braised, COX (IN...   
14   16807